# Exercise: Constrain Claude's Output with a System Prompt

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and split the setup into numbered sections so each code cell has a matching explanation.

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

`load_dotenv()` reads `ANTHROPIC_API_KEY` from a local `.env` file into the process environment. The **`Anthropic()`** client picks it up automatically, so the key stays out of the notebook source.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

Instantiate the client once and pin the **`model`**. Every request below reuses both.

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

Small wrappers keep the conversation loop readable. `add_user_message` and `add_assistant_message` append the right role dicts, and `chat` sends the list. The key detail: `chat` takes an **optional `system`** argument and only adds `system` to the request when you pass one. That is the lever this exercise pulls.

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Send a request with the code-only system prompt

The **system prompt** sets the persona and the output contract in one place: *return code only, no other text*. The user message asks for a specific function. Because the constraint lives in `system`, not in the user turn, it holds across every follow-up in the same conversation.

> The `system` value is a top-level parameter on `messages.create()`, not a message in the `messages` list. Passing it as a `role: "system"` dict is a common mistake and Claude will not treat it as the system prompt.

In [5]:
system_prompt = """
 You are a code assistant. Return the code content only. Don't include any other text.
"""

messages = []

add_user_message(messages, "Write a Python function that checks a string for duplicate characters.")

response = chat(messages, system=system_prompt)
add_assistant_message(messages, response)

response

'\n```python\ndef check_for_duplicates(string):\n    """\n    Check if a string contains duplicate characters.\n    \n    Args:\n        string: The input string to check\n        \n    Returns:\n        A dictionary with:\n        - \'has_duplicates\': Boolean indicating if duplicates exist\n        - \'duplicate_chars\': List of characters that appear more than once\n        - \'char_counts\': Dictionary with count of each character\n    """\n    char_counts = {}\n    \n    # Count occurrences of each character\n    for char in string:\n        char_counts[char] = char_counts.get(char, 0) + 1\n    \n    # Find characters that appear more than once\n    duplicate_chars = [char for char, count in char_counts.items() if count > 1]\n    \n    return {\n        \'has_duplicates\': len(duplicate_chars) > 0,\n        \'duplicate_chars\': duplicate_chars,\n        \'char_counts\': char_counts\n    }\n\n\nif __name__ == "__main__":\n    # Test cases\n    test_strings = [\n        "hello",\n  

### 6. Inspect the conversation history

The `messages` list now holds the **user question** and Claude's **assistant reply**. Note that the system prompt is *not* in this list - it is carried separately on each request. Read the assistant `content` and confirm it is bare code with no surrounding prose.

In [6]:
messages

[{'role': 'user',
  'content': 'Write a Python function that checks a string for duplicate characters.'},
 {'role': 'assistant',
  'content': '\n```python\ndef check_for_duplicates(string):\n    """\n    Check if a string contains duplicate characters.\n    \n    Args:\n        string: The input string to check\n        \n    Returns:\n        A dictionary with:\n        - \'has_duplicates\': Boolean indicating if duplicates exist\n        - \'duplicate_chars\': List of characters that appear more than once\n        - \'char_counts\': Dictionary with count of each character\n    """\n    char_counts = {}\n    \n    # Count occurrences of each character\n    for char in string:\n        char_counts[char] = char_counts.get(char, 0) + 1\n    \n    # Find characters that appear more than once\n    duplicate_chars = [char for char, count in char_counts.items() if count > 1]\n    \n    return {\n        \'has_duplicates\': len(duplicate_chars) > 0,\n        \'duplicate_chars\': duplicate_cha

### 7. Reference answer (commented out)

The cell below holds a **reference implementation** of the function you prompted for, kept commented out so it does not run. Uncomment it to compare Claude's returned code against a hand-written version.

In [7]:
#def has_duplicate_characters(s):
#   """Check if a string contains duplicate characters.
#      Args: 
#        s: String to check
#      Returns:
#        bool: True if duplicates exist, False otherwise
#    """
#    return len(s) != len(set(s))